[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Cypher](Cypher.md) | [Tasks](Task.md) | [Task 1](Task-1.md) | [Task 2](Task-2.md) | [Task 3](Task-3.md) | Notebook

# Exploring the Internet Yellow Pages (IYP)

Replace every `# YOUR CODE HERE` cell with working code, replace every *Replace this line with your answer.* prompt with your write-up, and note the run date printed by the setup cell alongside your numbers. Run the whole notebook top to bottom before you submit.

## Task 0: Setup and Data Access

This module only ever targets the public IYP instance — read-only, no credentials.

In [ ]:
import re
from datetime import date

import pandas as pd
from neo4j import GraphDatabase

IYP_URI = "neo4j://iyp-bolt.ihr.live:7687"
db = GraphDatabase.driver(IYP_URI, auth=None)
db.verify_connectivity()

# Never send a write. IYP is a shared read-only service, and this check catches the
# accidental case -- it is a courtesy, not a security boundary.
WRITE_KEYWORDS = ("CREATE", "MERGE", "SET", "DELETE", "DETACH", "DROP", "REMOVE", "LOAD CSV")
_COMMENT = re.compile(r"//[^\n]*")


def run_query(cypher, **params):
    """Run a read-only Cypher query and return the result as a pandas DataFrame."""
    # Strip // comments before checking: a comment saying "merge sources that
    # disagree" is prose, not a write, and matching on whole words keeps a
    # column named `num_created` from tripping the CREATE check.
    body = _COMMENT.sub("", cypher).upper()
    for keyword in WRITE_KEYWORDS:
        if re.search(rf"\b{keyword}\b", body):
            raise ValueError(f"refusing to run a query containing {keyword!r}: IYP is read-only")
    records, _, keys = db.execute_query(cypher, **params)
    return pd.DataFrame(records, columns=keys)


def show(df, title, limit=None):
    """Print a titled table, noting how many rows were withheld."""
    print(f"--- {title} ({len(df)} rows) ---")
    if df.empty:
        print("(no rows)")
    else:
        shown = df if limit is None else df.head(limit)
        print(shown.to_string(index=False))
        if limit is not None and len(df) > limit:
            print(f"... {len(df) - limit} more rows")
    print()


RUN_DATE = date.today().isoformat()
print(f"connected to {IYP_URI}")
print(f"run date     {RUN_DATE}   <- report this alongside your numbers")

Confirm the connection actually returns data before going further — a one-hop `RANK` lookup for a single AS, the same shape [Cypher §1](Cypher.md#1-running-cypher-from-python) and [Task 1](Task-1.md#canonical-names-and-caida-asrank) both already show.

In [ ]:
# YOUR CODE HERE: Task0 -- confirm the connection returns data

---

### Task 1.1 — Canonical names and CAIDA ASRank

In [ ]:
TASK1_ASNS = [6461, 2906, 4837]  # Zayo, Netflix, China Unicom

In [ ]:
# YOUR CODE HERE: Q1.a -- resolve a canonical name across sources, plus ASRank

**Q1.a** Do PeeringDB, BGP.Tools, and RIPE NCC agree on each AS's name? What does agreement or disagreement tell you about why IYP models `NAME` as multiple relationships instead of a single property?

*Your answer for Q1.a:*

Replace this line with your answer.

### Task 1.2 — IXP membership

In [ ]:
# YOUR CODE HERE: Q1.b -- top-10 IXPs globally, and per-AS IXP membership

**Q1.b** Which 10 IXPs have the most AS members globally? Separately, how many IXPs does each of the three running-example ASes belong to? How does each AS's count compare to the global top 10, and is IXP membership something all large networks do, or does it vary by the kind of network (transit ISP vs. content provider vs. national gateway)?

*Your answer for Q1.b:*

Replace this line with your answer.

### Task 1.3 — Peering degree, and putting all three metrics together

In [ ]:
# YOUR CODE HERE: Q1.c -- distinct AS peers, then combine with names_df/ixp_per_as_df

**Q1.c** How many distinct ASes does each of the three peer with directly (`PEERS_WITH` — every BGP peering session, not only settlement-free peering agreements)? Do ASRank, peering degree, and IXP membership move together, or does one stand out? What does that tell you about what CAIDA ASRank is actually measuring?

*Your answer for Q1.c:*

Replace this line with your answer.

---

### Task 2.1 — Popular hostnames inside AS2497 vs. AS6461

In [ ]:
HOSTNAME_ASNS = [2497, 6461]  # IIJ vs. Zayo -- content host vs. pure transit backbone

# YOUR CODE HERE: Q2.a -- popular hostnames per AS, pinned to Cisco Umbrella

# YOUR CODE HERE: Q2.a-density -- add a per-prefix density column

**Q2.a** For AS2497 and AS6461, how many distinct popular hostnames resolve into each AS's announced prefixes? Report both the raw count and the count per announced prefix. Does the gap match what you'd expect from a network that hosts content versus a pure transit backbone?

*Your answer for Q2.a:*

Replace this line with your answer.

### Task 2.2 — Authoritative nameservers inside AS2501

In [ ]:
NAMESERVER_ASN = 2501

# YOUR CODE HERE: Q2.b -- nameservers inside AS2501's address space, by domain count

# YOUR CODE HERE: Q2.b-concentration -- how concentrated is domain hosting

**Q2.b** For AS2501, which authoritative nameservers manage the most domains hosted inside its address space? Is domain hosting concentrated in a handful of nameservers, or spread evenly?

*Your answer for Q2.b:*

Replace this line with your answer.

**Q2.c** This traversal joins BGP origin, prefix containment, and DNS resolution in one Cypher query. `nids-dns-ecosystem` answers the same kind of question by loading OpenINTEL DNS measurements and a separately-obtained BGP prefix-to-AS mapping, then joining them in Spark. What did one Cypher query do here that took multiple stages there? What does the manual approach give you that the graph traversal doesn't?

*Your answer for Q2.c:*

Replace this line with your answer.

---

### Task 3.1 — ROA coverage for AS6461

In [ ]:
ROA_ASN = 6461

# YOUR CODE HERE: Q3.a -- ROAs for AS6461 with no exact-matching observed BGPPrefix

# Watch for 0.0.0.0/0 and 2000::/3: every AS that originates a global default route makes
# "at least one BGPPrefix inside" trivially true for every row unless you exclude those two roots.
DEFAULT_ROUTES = ["0.0.0.0/0", "2000::/3"]

# YOUR CODE HERE: Q3.a-detail -- for each unmatched ROA, is there a BGPPrefix nested inside it at all

**Q3.a** For AS6461, find every RPKI ROA (`ROUTE_ORIGIN_AUTHORIZATION`) that has no exact matching observed `BGPPrefix`. Looking at the actual prefixes, what's a plausible explanation for each?

*Your answer for Q3.a:*

Replace this line with your answer.

### Task 3.2 — Global RPKI-Invalid scan

In [ ]:
# YOUR CODE HERE: Q3.b -- top-20 ASes by RPKI-Invalid prefix count

EXAMPLE_ASNS = [6461, 2906, 4837, 2497]  # Task 1's three plus Task 2's IIJ

# YOUR CODE HERE: Q3.b-cross-ref -- where do the running examples fall on this list

**Q3.b** Across the whole graph, which 20 ASes originate the most prefixes tagged `"RPKI Invalid"`? Do any of the four running-example ASes (AS6461, AS2906, AS4837, AS2497) appear on that list? If not, what does that suggest about them? If so, does it look like a genuine anomaly or something explainable?

*Your answer for Q3.b:*

Replace this line with your answer.

### Task 3.3 — What does "RPKI Invalid" actually mean?

In [ ]:
# YOUR CODE HERE: Q3.c -- break the RPKI-Invalid tag down by its exact label

**Q3.c** Does an `"RPKI Invalid"` tag prove a prefix hijack? What are at least two other explanations for the same tag? How does this graph-pattern comparison relate to the manual IRR/RPKI/BGP cross-comparison in [`nids-irr-rpki-whois`](https://github.com/CAIDA/nids-irr-rpki-whois) -- what does the graph give you "for free" that you had to build there, and what nuance (if any) does the graph pattern flatten away?

*Your answer for Q3.c:*

Replace this line with your answer.

---

In [ ]:
db.close()
print("connection closed")

[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Cypher](Cypher.md) | [Tasks](Task.md) | [Task 1](Task-1.md) | [Task 2](Task-2.md) | [Task 3](Task-3.md) | Notebook